In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_PATH = Path('uganda_drug_supply_synthetic.csv')

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print('shape:', df.shape)
display(pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str)}))

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
duplicates = int(df.duplicated().sum())
invalid_negatives = {c: int((df[c] < 0).sum()) for c in df.select_dtypes(include=[np.number]).columns}
display(missing_pct.to_frame('missing_pct'))
print('duplicates:', duplicates)
print('negative_count_by_numeric_col:', invalid_negatives)

In [ ]:
display(df.select_dtypes(include='number').describe().T)

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f'\n{col}')
    display(df[col].value_counts(dropna=False).head(20))

In [ ]:
tmp = df.copy()
tmp['stock_received_date'] = pd.to_datetime(tmp['stock_received_date'], errors='coerce')
weekly = tmp.set_index('stock_received_date').resample('W').size()
weekly.plot(title='Weekly record volume', figsize=(12, 4))

In [ ]:
pivot = df.pivot_table(index='distribution_region', values=['average_monthly_demand', 'stockout_occurred', 'expiry_rate_percent'], aggfunc='mean')
display(pivot.sort_values('average_monthly_demand', ascending=False))

In [ ]:
from ml.feature_spec import FORBIDDEN_COLUMNS
leak_cols = [c for c in FORBIDDEN_COLUMNS if c in df.columns]
display(pd.DataFrame({'flagged_column': leak_cols}))

In [ ]:
from config import ARTIFACTS_DIR
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
report = {
    'shape': df.shape,
    'missing_pct': (df.isna().mean() * 100).to_dict(),
    'duplicates': int(df.duplicated().sum()),
    'flagged_leakage_columns': leak_cols,
}
pd.Series(report).to_json(ARTIFACTS_DIR / 'eda_report.json', indent=2)
print('Saved', ARTIFACTS_DIR / 'eda_report.json')